# Release FP2026.1: CH-LAE fluxes and meteo, 2016-2026.09

**notebook version**: `1` (27 Oct 2026)

## ℹ️ About this notebook

This notebook assembles the dataset of the release **FP2026.1**, `CH-LAE_FP2026.1_2016-2026.09`. It
covers 1 January 2016 to 30 September 2026 and is cut from the collection
`10_FLUXES_and_METEO_2004-2026.09`, which holds all data available so far on one 30-minute timestamp
from 2004 on.

The cut keeps the records of the release period and the variables that have at least one value in
it. Variables whose record ends before 2016 are therefore not part of the release.

The dataset is written as parquet and as CSV. The last section lists every variable of the dataset
with its unit, a short description, its source and its period. The units and descriptions come from
`variable_info.py` in this folder.

## ⚙️ Settings

In [1]:
DATADIR = (r"F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data"
           r"\workflow")
COLLECTION = rf"{DATADIR}\80_FINALIZE\10_FLUXES_and_METEO_2004-2026.09.parquet"
# The source files of the collection, in the order it joins them; read for the column names only.
SOURCES = {
    'L1 fluxes IRGA72': rf"{DATADIR}\20_L1_FLUXES\IRGA72\99_IRGA72_L1_MERGED_2016-2026.09.parquet",
    'meteo': rf"{DATADIR}\10_METEO\30_PRODUCTS\99_METEO_MERGED_2004-2026.09.parquet",
    'FPPC IRGA72': (rf"{DATADIR}\30_FLUX_POST-PROCESSING_CHAIN\IRGA72"
                    r"\99_FPPC_IRGA72_MERGED_2016-2026.09.parquet"),
}
# The collection renames EddyPro's potential radiation of the Level-1 fluxes.
RENAMED = {'L1 fluxes IRGA72': {'SW_IN_POT': 'SW_IN_POT_EDDYPRO'}}
RELEASE = 'FP2026.1'
PERIOD_START, PERIOD_END = '2016-01-01 00:15', '2026-09-30 23:45'

OUTNAME = "20_CH-LAE_FP2026.1_2016-2026.09"  # same name as this notebook
OUTPATH = rf"{DATADIR}\80_FINALIZE"

In [2]:
from datetime import datetime
import importlib.metadata
from pathlib import Path

import diive as dv
import pandas as pd
import pyarrow.parquet as pq

from variable_info import describe_all, info_section

NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive version: v{importlib.metadata.version('diive')}")

Last run: 2026-10-27 13:00:52
diive version: v0.93.0


## 📂 Load data

In [3]:
collection = dv.load_parquet(filepath=COLLECTION)
assert collection.index.is_unique, "the collection has duplicate timestamps"
print(f"Collection: {collection.shape[0]:,} records x {collection.shape[1]} columns, "
      f"{collection.index[0]} -> {collection.index[-1]}")

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\80_FINALIZE\10_FL
UXES_and_METEO_2004-2026.09.parquet (0.538 seconds).

Collection: 398,832 records x 548 columns, 2004-01-01 00:15:00 -> 2026-09-30 23:45:00


The source of each column, as the collection assigns it: a column belongs to the first source file
that contains it.

In [4]:
SOURCE = {}
for _name, _path in SOURCES.items():
    for _c in pq.read_schema(_path).names:
        _c = RENAMED.get(_name, {}).get(_c, _c)
        if _c in collection.columns and _c not in SOURCE:
            SOURCE[_c] = _name
_missing = collection.columns.difference(list(SOURCE))
assert _missing.empty, f"columns without a source: {list(_missing)}"

## ✂️ Cut the release period

In [5]:
expected = pd.date_range(PERIOD_START, PERIOD_END, freq='30min', name=collection.index.name)
dataset = collection.loc[PERIOD_START:PERIOD_END]
assert dataset.index.equals(expected), f"the collection does not cover {PERIOD_START} to {PERIOD_END} in 30 min"

# Keep only the variables with at least one value in the release period.
_empty = dataset.columns[dataset.isna().all()]
dataset = dataset.drop(columns=_empty)
SOURCE_ALL = SOURCE
SOURCE = {c: SOURCE_ALL[c] for c in dataset.columns}
DESCRIBED = describe_all(dataset.columns)  # unit and description of every column
print(f"{RELEASE}: {dataset.shape[0]:,} records x {dataset.shape[1]} columns, "
      f"{dataset.index[0]} -> {dataset.index[-1]}")
print(f"Dropped {len(_empty)} variables without a value in the release period")

FP2026.1: 188,448 records x 548 columns, 2016-01-01 00:15:00 -> 2026-09-30 23:45:00
Dropped 0 variables without a value in the release period


### 📋 Table: variables dropped

In [6]:
_dropped = pd.DataFrame({
    'source': [SOURCE_ALL[c] for c in _empty],
    'last': collection[_empty].apply(lambda s: s.last_valid_index()),
}, index=pd.Index(_empty, name='variable'))
with pd.option_context('display.max_rows', None):
    display(_dropped)

,source,last
variable,,


### 📋 Table: variables per source

In [7]:
display(pd.Series(SOURCE).value_counts().reindex(list(SOURCES)).rename('variables').to_frame())

,variables
L1 fluxes IRGA72,329
meteo,61
FPPC IRGA72,158


## 💾 Export

The dataset as parquet and as CSV, and next to them an `_INFO.md` file that lists every variable with
its unit and a short description, per source. In the CSV, the timestamp is the first column, `TIMESTAMP_MIDDLE`, the middle of each
30-minute record in UTC+01:00; missing values are empty fields.

In [8]:
filepath = dv.save_parquet(data=dataset, filename=OUTNAME, outpath=OUTPATH)
csvpath = Path(OUTPATH) / f"{OUTNAME}.csv"
dataset.to_csv(csvpath, date_format='%Y-%m-%d %H:%M:%S')
print(f"Wrote {csvpath} ({csvpath.stat().st_size / 1e6:,.0f} MB)")

# The CSV reads back to the same data.
_back = pd.read_csv(csvpath, index_col=0, parse_dates=True)
assert _back.index.equals(dataset.index) and _back.columns.equals(dataset.columns)
_num = dataset.select_dtypes('number').columns
assert ((_back[_num] - dataset[_num]).abs().max() < 1e-9).all(), "the CSV does not match the data"
del _back

> Saved file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\80_FINALIZE\20_CH
-LAE_FP2026.1_2016-2026.09.parquet (4.252 seconds).

Wrote F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\80_FINALIZE\20_CH-LAE_FP2026.1_2016-2026.09.csv (688 MB)


In [9]:
_lines = [f"# {OUTNAME}", "",
          f"Release {RELEASE}, {dataset.index[0]} to {dataset.index[-1]} ({dataset.index.name}, UTC+01:00), "
          f"{dataset.shape[0]:,} records x {dataset.shape[1]} variables.",
          f"Written {datetime.now():%Y-%m-%d %H:%M} by `{OUTNAME}.ipynb` from `{Path(COLLECTION).name}`.", "",
          "| source | file |", "|---|---|"]
_lines += [f"| {k} | `{v}` |" for k, v in SOURCES.items()]
for _name in SOURCES:
    _lines += info_section(_name, [c for c, s in SOURCE.items() if s == _name], DESCRIBED)
with open(rf"{OUTPATH}\{OUTNAME}_INFO.md", "w", encoding="utf-8") as _f:
    _f.write("\n".join(_lines) + "\n")
print(f"Wrote {OUTPATH}\\{OUTNAME}_INFO.md")

Wrote F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\80_FINALIZE\20_CH-LAE_FP2026.1_2016-2026.09_INFO.md


## 📋 Variables in the dataset

Every variable of the dataset, in the order of the file: its unit, a short description, its source,
the number and share of records with a value, and the first and last of these records.

### 📋 Table: all variables

In [10]:
_valid = dataset.notna()
VARIABLES = pd.DataFrame({
    'unit': pd.Series({c: u for c, (u, d) in DESCRIBED.items()}),
    'description': pd.Series({c: d for c, (u, d) in DESCRIBED.items()}),
    'source': pd.Series(SOURCE),
    'records': _valid.sum(),
    'coverage (%)': (_valid.mean() * 100).round(1),
    'first': dataset.apply(lambda s: s.first_valid_index()),
    'last': dataset.apply(lambda s: s.last_valid_index()),
}).reindex(dataset.columns)
VARIABLES.index.name = 'variable'
VARIABLES.insert(0, '#', range(1, len(VARIABLES) + 1))
print(f"{len(VARIABLES)} variables")
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(VARIABLES)

548 variables


,#,unit,description,source,records,coverage (%),first,last
variable,,,,,,,,
AIR_CP,1,J kg-1 K-1,Specific heat of moist air at constant pressure,L1 fluxes IRGA72,179251,95.1,2016-01-11 15:45:00,2026-09-30 23:45:00
AIR_DENSITY,2,kg m-3,Density of moist air,L1 fluxes IRGA72,179251,95.1,2016-01-11 15:45:00,2026-09-30 23:45:00
AIR_MV,3,m3 mol-1,Molar volume of moist air,L1 fluxes IRGA72,179251,95.1,2016-01-11 15:45:00,2026-09-30 23:45:00
AIR_RHO_CP,4,J m-3 K-1,Volumetric heat capacity of moist air,L1 fluxes IRGA72,179251,95.1,2016-01-11 15:45:00,2026-09-30 23:45:00
AOA_METHOD,5,-,Angle-of-attack correction (EddyPro code),L1 fluxes IRGA72,179251,95.1,2016-01-11 15:45:00,2026-09-30 23:45:00
AXES_ROTATION_METHOD,6,-,Coordinate rotation method (EddyPro code),L1 fluxes IRGA72,179251,95.1,2016-01-11 15:45:00,2026-09-30 23:45:00
BADM_HEIGHTC,7,m,Canopy height,L1 fluxes IRGA72,179251,95.1,2016-01-11 15:45:00,2026-09-30 23:45:00
BADM_INSTPAIR_EASTWARD_SEP_GA_CO2,8,cm,"Eastward separation of the analyser from the sonic, CO2",L1 fluxes IRGA72,174863,92.8,2016-01-22 12:15:00,2026-09-30 23:45:00
BADM_INSTPAIR_EASTWARD_SEP_GA_H2O,9,cm,"Eastward separation of the analyser from the sonic, H2O",L1 fluxes IRGA72,174988,92.9,2016-01-12 17:15:00,2026-09-30 23:45:00


***

# End of notebook.

In [11]:
NOTEBOOK_END = datetime.now()
print(f"Runtime: {NOTEBOOK_END - NOTEBOOK_START}")

Runtime: 0:01:14.332464
